## Validity Tests For Weight Optimization
Post-tuning validity and visualizations

In [ ]:
import os

import kaleido
print(kaleido.__file__)

from ax.adapter.cross_validation import cross_validate, compute_diagnostics
from ax.adapter.registry import Generators
from ax.plot.diagnostic import interact_cross_validation
from ax.plot.slice import plot_slice
from ax.storage.sqa_store.db import init_engine_and_session_factory
from ax.storage.sqa_store.load import load_experiment
from ax.utils.notebook.plotting import render
from pprint import pprint
from ax.plot.scatter import interact_fitted
from ax.utils.notebook.plotting import init_notebook_plotting
from ax.plot.contour import interact_contour
import plotly.graph_objects as go
import plotly.io as pio

## Configuration
Set the consts if needed

In [ ]:
pio.templates.default = "plotly_white"
EXP_NAME = "Weight Optimization For Monolith Decomposition"
DB_URL = "sqlite:///ax.sqlite"
VISU_FOLDER = "visualizations"

In [ ]:
init_engine_and_session_factory(DB_URL)
exp = load_experiment(EXP_NAME)

data = exp.fetch_data()
model = Generators.BOTORCH_MODULAR(experiment=exp, data=data)
init_notebook_plotting()
os.makedirs(VISU_FOLDER, exist_ok=True)

## Visualization

In [ ]:
render(interact_contour(model, metric_name="decomposition_metric_mean", lower_is_better=True))

In [ ]:
render(interact_fitted(model, rel=False))

## Cross Validation Results

In [ ]:
cv = cross_validate(model)
pprint(compute_diagnostics(cv))

In [ ]:
fig = interact_cross_validation(cv)
render(fig)

go_fig = go.Figure(data=fig.data)
pio.write_image(go_fig, f"{VISU_FOLDER}/cross-validation.pdf", scale=2)



### Slice Plots

In [ ]:

fig = plot_slice(model, "w_calls", "decomposition_metric_mean")
render(fig)

go_fig = go.Figure(data=fig.data)
go_fig.update_layout(
    xaxis_title=r"$w_{Calls}$",
    yaxis_title=r"$\text{Predicated }f(w_{Calls}, \cdots)"
)
pio.write_image(go_fig, f"{VISU_FOLDER}/w_calls_curve.pdf", scale=2)



In [ ]:

fig = plot_slice(model, "precision_1", "decomposition_metric_mean")
render(fig)

go_fig = go.Figure(data=fig.data)
go_fig.update_layout(
    xaxis_title=r"$\gamma_1$",
    yaxis_title=r"$f(\gamma_1, \cdots)$"
)
pio.write_image(go_fig, "smoothness.pdf", scale=2)



In [ ]:
from ax.storage.sqa_store.structs import DBSettings
from ax.service.ax_client import AxClient

db_settings = DBSettings(url=DB_URL)
client = AxClient(
    db_settings=db_settings
)
client.load_experiment_from_database(EXP_NAME)
client.compute_analyses(display=True)